# Unified feature extraction

Single notebook for all feature batteries. Uses **eyefeatures.data** (Parquet + meta) and shared **collection_experiments** utils.

**Parts:**
1. **Simple** – fixation duration, saccade length (mean, std, min, max, count, sum)
2. **Extended** – extended stats, regression, angles
3. **Complex** – Hurst, entropies, RQA, Lyapunov, fractal, HHT, etc. (full hyperparameter grids)
4. **Distance** – split-first pipeline; fit on train only; path_pk per split_id via PATH_PK_PER_LABEL

Run **0_create_splits** first. Outputs go to `features_output/` and `features_output/splits/`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from eyefeatures.features.extractor import Extractor
from eyefeatures.features.stats import FixationFeatures, SaccadeFeatures, RegressionFeatures
from eyefeatures.features.measures import (
    HurstExponent, SpectralEntropy, FuzzyEntropy, SampleEntropy,
    IncrementalEntropy, GriddedDistributionEntropy,
    PhaseEntropy, LyapunovExponent, FractalDimension, CorrelationDimension,
    RQAMeasures, SaccadeUnlikelihood, HHTFeatures,
)

from utils.benchmark_utils import (
    get_collection_dir,
    list_datasets,
    load_dataset_with_meta,
    col_info_from_meta,
    get_split_info_paths_for_dataset,
)
from utils.feature_extraction_utils import setup_paths, extract_and_save_features, print_summary
from utils.distance_extraction_utils import (
    extract_and_save_distance_features,
    SIMPLE_DISTANCE_METHODS,
    ADVANCED_DISTANCE_METHODS,
    EXPECTED_PATH_METHODS,
)

In [ ]:
# Shared setup
paths = setup_paths(output_dir='features_output', splits_dir='features_output/splits')
COLLECTION_DIR = paths['collection_dir']
dataset_names = list_datasets(dataset_type='fixation')
print(f"Datasets: {len(dataset_names)}")
print(f"Output: {paths['output_dir']}")

## Part 1: Simple features

Fixation duration and saccade length statistics (mean, std, min, max, count, sum).

In [ ]:
basic_stats = ['mean', 'std', 'min', 'max', 'count', 'sum']
results_simple = []

for dataset_name in dataset_names:
    print(f"\n{'='*80}\n[Simple] {dataset_name}\n{'='*80}")
    try:
        df, meta_info = load_dataset_with_meta(dataset_name)
        col_info = col_info_from_meta(df, meta_info)
        if not col_info['group_cols'] or not col_info.get('duration'):
            results_simple.append({'dataset': dataset_name, 'status': 'skipped', 'reason': 'missing pk or duration'})
            continue
        fix = FixationFeatures(features_stats={'duration': basic_stats}, x=col_info['x_col'], y=col_info['y_col'], duration='duration', pk=col_info['group_cols'], return_df=True)
        sac = SaccadeFeatures(features_stats={'length': basic_stats}, x=col_info['x_col'], y=col_info['y_col'], t=col_info['t_col'], duration='duration', pk=col_info['group_cols'], return_df=True)
        extractor = Extractor(features=[fix, sac], x=col_info['x_col'], y=col_info['y_col'], t=col_info['t_col'], duration='duration', pk=col_info['group_cols'], return_df=True)
        result = extract_and_save_features(df, dataset_name, 'simple_features', extractor, meta_info, paths, check_cache_first=True)
        results_simple.append(result)
    except Exception as e:
        import traceback
        traceback.print_exc()
        results_simple.append({'dataset': dataset_name, 'status': 'error', 'error': str(e)})

print_summary(results_simple, feature_type='simple_features')

## Part 2: Extended statistical features

Extended stats (median, skew, kurtosis), fixation duration, saccade length/speed/acceleration/angles, regression features.

In [ ]:
extended_stats = ['mean', 'std', 'min', 'max', 'count', 'median', 'skew', 'kurtosis']
results_extended = []

for dataset_name in dataset_names:
    print(f"\n{'='*80}\n[Extended] {dataset_name}\n{'='*80}")
    try:
        df, meta_info = load_dataset_with_meta(dataset_name)
        col_info = col_info_from_meta(df, meta_info)

        pk = col_info.get('group_cols')
        if not pk:
            results_extended.append({'dataset': dataset_name, 'status': 'skipped', 'reason': 'missing pk'})
            continue

        x_col = col_info.get('x_col')
        y_col = col_info.get('y_col')

        # Prefer meta-provided t; otherwise fall back to start_time if present.
        t_col = col_info.get('t_col')
        if not t_col and 'start_time' in df.columns:
            t_col = 'start_time'
        has_t = bool(t_col)

        has_duration = bool(col_info.get('duration'))

        # dt computation inside saccade/regression stats requires a valid t column.
        # Only pass duration through when t exists.
        duration_col = 'duration' if (has_duration and has_t) else None

        features_list = []

        if has_duration:
            fix = FixationFeatures(
                features_stats={'duration': extended_stats + ['sum']},
                x=x_col,
                y=y_col,
                duration="duration",
                pk=pk,
                return_df=True,
            )
            features_list.append(fix)

        sac_stats = {'length': extended_stats + ['sum']}
        if has_t:
            sac_stats['direction_angle'] = extended_stats
            sac_stats['rotation_angle'] = extended_stats
        # Requirement: do NOT calculate speed/acceleration if duration does not exist.
        if has_duration and has_t:
            sac_stats['speed'] = extended_stats
            sac_stats['acceleration'] = extended_stats

        sac = SaccadeFeatures(
            features_stats=sac_stats,
            x=x_col,
            y=y_col,
            t=t_col,
            duration="duration",
            pk=pk,
            return_df=True,
        )
        features_list.append(sac)

        # RegressionFeatures computes dt for any non-length feature (including mask),
        # so only include it when a valid t column is available.
        if has_t:
            reg_stats = {
                'length': extended_stats,
                'mask': ['sum'],
            }
            if has_duration:
                reg_stats['speed'] = extended_stats

            reg = RegressionFeatures(
                features_stats=reg_stats,
                x=x_col,
                y=y_col,
                t=t_col,
                duration="duration",
                pk=pk,
                return_df=True,
            )
            features_list.append(reg)

        extractor = Extractor(
            features=features_list,
            x=x_col,
            y=y_col,
            t=t_col,
            duration="duration",
            pk=pk,
            return_df=True,
        )
        result = extract_and_save_features(
            df,
            dataset_name,
            'extended_features',
            extractor,
            meta_info,
            paths,
            check_cache_first=True,
        )
        results_extended.append(result)
    except Exception as e:
        import traceback

        traceback.print_exc()
        results_extended.append({'dataset': dataset_name, 'status': 'error', 'error': str(e)})

print_summary(results_extended, feature_type='extended_features')


## Part 3: Complex features

Full hyperparameter grids: Hurst, SpectralEntropy, FuzzyEntropy, SampleEntropy, IncrementalEntropy, GriddedDistributionEntropy, PhaseEntropy, LyapunovExponent, FractalDimension, CorrelationDimension, RQAMeasures, SaccadeUnlikelihood, HHTFeatures.

In [ ]:
results_complex = []

for dataset_name in dataset_names:
    print(f"\n{'='*80}\n[Complex] {dataset_name}\n{'='*80}")
    try:
        df, meta_info = load_dataset_with_meta(dataset_name)
        col_info = col_info_from_meta(df, meta_info)
        if not col_info['group_cols'] or not col_info.get('duration'):
            results_complex.append({'dataset': dataset_name, 'status': 'skipped', 'reason': 'missing pk or duration'})
            continue
        pk, x_col, y_col, t_col = col_info['group_cols'], col_info['x_col'], col_info['y_col'], col_info['t_col']
        duration = 'duration' if col_info.get('has_duration') else None
        features_list = []
        for n_iters in [1, 2, 3, 4, 5]:
            for fill_strategy in ['last', 'mean', 'reduce']:
                features_list.append(HurstExponent(n_iters=n_iters, fill_strategy=fill_strategy, coordinate=x_col, pk=pk, return_df=True, ignore_errors=True))
        for n_iters in [1, 2, 3, 4, 5]:
            for fill_strategy in ['last', 'mean', 'reduce']:
                features_list.append(HurstExponent(n_iters=n_iters, fill_strategy=fill_strategy, coordinate=y_col, pk=pk, return_df=True, ignore_errors=True))
        features_list.append(SpectralEntropy(x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for m in [1, 2, 3]:
            for r in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]:
                features_list.append(FuzzyEntropy(m=m, r=r, x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for m in [1, 2, 3]:
            for r in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]:
                features_list.append(SampleEntropy(m=m, r=r, x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        features_list.append(IncrementalEntropy(x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for grid_size in [4, 5, 6, 7, 8, 16, 32]:
            features_list.append(GriddedDistributionEntropy(grid_size=grid_size, x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for m in [1, 2, 3, 4, 5]:
            for tau in [1, 2, 3, 4, 5]:
                features_list.append(PhaseEntropy(m=m, tau=tau, x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for m in [1, 2, 3, 4, 5]:
            for tau in [1, 2, 3, 4, 5]:
                features_list.append(LyapunovExponent(m=m, tau=tau, T=1, x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for m in [2, 3, 4]:
            for tau in [1, 2, 3, 4, 5]:
                features_list.append(FractalDimension(m=m, tau=tau, x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for m in [2, 3, 4]:
            for tau in [1, 2, 3, 4, 5]:
                for r in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]:
                    features_list.append(CorrelationDimension(m=m, tau=tau, r=r, x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for rho in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]:
            for min_length in [1, 2, 3, 4]:
                features_list.append(RQAMeasures(rho=rho, min_length=min_length, measures=['rec', 'det', 'lam', 'corm'], x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        features_list.append(SaccadeUnlikelihood(x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        for max_imfs in [-1] + list(range(1, 25, 5)):
            features_list.append(HHTFeatures(max_imfs=max_imfs, features=['mean', 'std', 'var', 'entropy', 'energy'], x=x_col, y=y_col, pk=pk, return_df=True, ignore_errors=True))
        extractor = Extractor(features=features_list, x=x_col, y=y_col, t=t_col, duration=duration, pk=pk, return_df=True)
        result = extract_and_save_features(df, dataset_name, 'complex_features', extractor, meta_info, paths, check_cache_first=True)
        results_complex.append(result)
    except Exception as e:
        import traceback
        traceback.print_exc()
        results_complex.append({'dataset': dataset_name, 'status': 'error', 'error': str(e)})

print_summary(results_complex, feature_type='complex_features')

## Part 4: Distance features

**Different pipeline**: split data first; fit transformers on **train only** (expected paths from train); transform train/val/test. path_pk (reference path grouping) per split_id via **PATH_PK_PER_LABEL**. Simple methods: euc, hau, dtw, man, eye, dfr. Advanced: tde, multimatch. Expected path methods: mean, fwp.

In [ ]:
# PATH_PK_PER_LABEL: path_pk per split_id
PATH_PK_PER_LABEL = {
    '3D_condition_label': ['group_category_label', 'group_filenumber'],
    '3D_group_category_label': ['group_subject'],
    'AFC_group_category_label': ['group_subject'],
    'AFC_mod_label': ['group_category_label', 'group_filenumber'],
    'AFC_scr_label': ['group_category_label', 'group_filenumber'],
    'APPC_context_group_category_label': ['group_subject'],
    'APP_group_category_label': ['group_subject'],
    'APP_known_label': ['group_category_label', 'group_filenumber'],
    'APP_meta_certainty_label': ['group_category_label', 'group_filenumber'],
    'APP_mod_label': ['group_category_label', 'group_filenumber'],
    'ASD_fixations_ASD_label': ['group_observation'],
    'Age_study_group_category_label': ['group_subject'],
    'Age_study_meta_age_label': ['group_category_label', 'group_filenumber'],
    'Baseline_group_category_label': ['group_subject'],
    'Bias_group_category_label': ['group_subject'],
    'Crossmodal2_group_category_label': ['group_subject'],
    'Crossmodal_condition_label': ['group_category_label', 'group_filenumber'],
    'Crossmodal_group_category_label': ['group_subject'],
    'Dyslexia_1_fixations_Dyslexia_label': ['group_stimul'],
    'Dyslexia_1_fixations_meta_Age_label': ['group_stimul'],
    'Dyslexia_1_fixations_meta_Sex_label': ['group_stimul'],
    'Dyslexia_2_fixations_Dyslexia_label': ['group_stimul'],
    'Dyslexia_2_fixations_meta_Age_label': ['group_stimul'],
    'Dyslexia_Czech_fixations_Dyslexia_label': ['group_task'],
    'Filtered_group_category_label': ['group_subject'],
    'Filtered_meta_delay_label': ['group_category_label', 'group_filenumber'],
    'Filtered_meta_spatial_filter_label': ['group_category_label', 'group_filenumber'],
    'Gap_gap_label': ['group_category_label', 'group_filenumber'],
    'Gap_group_category_label': ['group_subject'],
    'Head_fixed_group_category_label': ['group_subject'],
    'Head_fixed_meta_condition_label': ['group_category_label', 'group_filenumber'],
    'Head_fixed_meta_guided_viewing_label': ['group_category_label', 'group_filenumber'],
    'Memory_1_group_category_label': ['group_subject'],
    'Memory_1_group_iteration_label': ['group_category_label', 'group_filenumber'],
    'Memory_2_condition_label': ['group_category_label', 'group_filenumber'],
    'Memory_2_group_category_label': ['group_category_label', 'group_filenumber'],
    'Memory_2_group_iteration_label': ['group_category_label', 'group_filenumber'],
    'Paris_experiment_fixations_TEXT_TYPE_2_label': ['group_subject'],
    'Paris_experiment_fixations_TEXT_TYPE_label': ['group_subject'],
    'Patch_group_category_label': ['group_subject'],
    'Patch_scr_label': ['group_category_label', 'group_filenumber'],
    'Surgical_skills_1_fixations_Performance_label': ['group_Task'],
    'Surgical_skills_2_fixations_Performance_label': ['group_Subject'],
    'Webtask_condition_label': ['group_category_label', 'group_filenumber'],
    'Webtask_familiarity_label': ['group_category_label', 'group_filenumber'],
    'Webtask_group_category_label': ['group_subject'],
    'Webtask_relevance_label': ['group_category_label', 'group_filenumber'],
    'Webtask_school_condition_label': ['group_category_label', 'group_filenumber'],
    'Webtask_school_familiarity_label': ['group_category_label', 'group_filenumber'],
    'Webtask_school_group_category_label': ['group_subject'],
    'Webtask_school_relevance_label': ['group_category_label', 'group_filenumber'],
    'Webtask_school_search_result_label': ['group_category_label', 'group_filenumber'],
    'Webtask_search_result_label': ['group_category_label', 'group_filenumber'],
}
for _name in dataset_names:
    try:
        _df, _meta = load_dataset_with_meta(_name)
        _ci = col_info_from_meta(_df, _meta)
        _pk = _ci.get('group_cols') or []
        if not _pk:
            continue
        for _path in get_split_info_paths_for_dataset(paths['splits_dir'], _name):
            _sid = _path.stem.replace('_split_info', '')
            PATH_PK_PER_LABEL.setdefault(_sid, list(_pk))
    except Exception:
        pass
print(f"PATH_PK_PER_LABEL: {len(PATH_PK_PER_LABEL)} split_ids")

In [ ]:
results_distance = []

for dataset_name in dataset_names:
    print(f"\n{'='*80}\n[Distance] {dataset_name}\n{'='*80}")
    try:
        df, meta_info = load_dataset_with_meta(dataset_name)
        col_info = col_info_from_meta(df, meta_info)
        pk = col_info.get('group_cols') or []
        if not pk:
            results_distance.append({'dataset': dataset_name, 'status': 'skipped', 'reason': 'no pk'})
            continue
        n_scanpaths = len(df[pk].drop_duplicates())
        if n_scanpaths < 2:
            results_distance.append({'dataset': dataset_name, 'status': 'skipped', 'reason': 'need at least 2 scanpaths'})
            continue
        results = extract_and_save_distance_features(
            df, dataset_name, meta_info, col_info, paths,
            simple_methods=SIMPLE_DISTANCE_METHODS,
            advanced_methods=ADVANCED_DISTANCE_METHODS,
            expected_path_methods=EXPECTED_PATH_METHODS,
            path_pk_per_label=PATH_PK_PER_LABEL,
            check_cache_per_split=True,
        )
        for r in results:
            r['dataset'] = dataset_name
            results_distance.append(r)
    except Exception as e:
        import traceback
        traceback.print_exc()
        results_distance.append({'dataset': dataset_name, 'status': 'error', 'error': str(e)})

print_summary(results_distance, feature_type='distance_features')